# Full Pipeline Test: STT + Text Emotion + Voice Emotion

This notebook batch-tests all three trained components together on the shared test
dataset (`audio_transcript/dataset`):

1. **STT**: `openai/whisper-large-v3` (audio -> transcript)
2. **Text emotion**: fine-tuned XLM-RoBERTa model, `ftmjfr/VEMO-final-text-emotion-recognition`
   on the Hugging Face Hub (transcript -> emotion)
3. **Voice emotion**: SHEMO Phase 1 encoder (`Wav2Vec2Model`, embeddings only) + **`final_classifier_v2`**
   logistic-regression head trained on those embeddings via leave-one-speaker-out CV
   (audio -> emotion, 4 classes only: anger, happiness, neutral, sadness -- no fear/surprise)

Text and voice predictions are each a full probability vector over the same
4 unified classes, combined as `(audio_probs + text_probs) / 2`. 

**Before running:**
- Enable GPU and Internet in notebook settings
- Update `DATASET_BASE_PATH` and `AUDIO_EMOTION_MODEL_PATH` below to match your
  actual Kaggle input paths


## 1. Install dependencies

In [47]:
# Install only the packages not already present in the Kaggle environment
# (avoid -U / upgrading torch, transformers, scikit-learn, librosa, soundfile --
#  they're already installed and version-pinned to match Kaggle's CUDA stack;
#  force-upgrading them breaks GPU package compatibility)
!pip install -q jiwer hazm openpyxl sentencepiece gdown

In [48]:
!gdown --folder "https://drive.google.com/drive/folders/1o76OBYskZQ-NMD3YozE_pxcPU7tn-Ioj" -O /kaggle/working/shemo_phase1_checkpoint


Retrieving folder contents
Processing file 1P0aoIOavT6Q2JyYy7mwxKCP-bHz5RGu_ config.json
Processing file 1-gpLOfGR71M7gFFD2Gk7mrRobq72R-cQ model.safetensors
Processing file 1nx8BhkT4euvTnQjCXoFn8abzTjbRf0Ca preprocessor_config.json
Processing file 1c-Mlwh8YT-wtmOsm_moVQUUhYMKv5He3 training_args.bin
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1P0aoIOavT6Q2JyYy7mwxKCP-bHz5RGu_
To: /kaggle/working/shemo_phase1_checkpoint/config.json
100%|██████████████████████████████████████| 2.58k/2.58k [00:00<00:00, 12.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1-gpLOfGR71M7gFFD2Gk7mrRobq72R-cQ
From (redirected): https://drive.google.com/uc?id=1-gpLOfGR71M7gFFD2Gk7mrRobq72R-cQ&confirm=t&uuid=96e3822d-f0f1-4570-aa51-d199fe7acdde
To: /kaggle/working/shemo_phase1_checkpoint/model.safetensors
100%|███████████████████████████████████████| 1.27G/1.27G [00:10<00:00, 126MB

In [49]:

!gdown --folder "https://drive.google.com/drive/folders/1CHpEGR0y0MpAzaEUq5KEB6yjIMeFi1Jy" -O /kaggle/working/final_classifier_v2

Retrieving folder contents
Processing file 1c2XkAlE0UdfzwZVvbgPyVp90sPxQzkAw logistic_classifier.joblib
Processing file 1qfr_phSdZdXeZuf-KrA0Pbe_mBpZOgSc metadata.json
Processing file 1MMWCyDwsz2g7RtNoygKLtYDxMBRzVGM5 scaler.joblib
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1c2XkAlE0UdfzwZVvbgPyVp90sPxQzkAw
To: /kaggle/working/final_classifier_v2/logistic_classifier.joblib
100%|██████████████████████████████████████| 34.1k/34.1k [00:00<00:00, 63.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1qfr_phSdZdXeZuf-KrA0Pbe_mBpZOgSc
To: /kaggle/working/final_classifier_v2/metadata.json
100%|██████████████████████████████████████| 35.2k/35.2k [00:00<00:00, 17.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MMWCyDwsz2g7RtNoygKLtYDxMBRzVGM5
To: /kaggle/working/final_classifier_v2/scaler.joblib
100%|██████████████████████████████████████| 25.4k/25.4k [00:00<00:0

## 2. Imports and configuration

In [50]:
!pip install pyloudnorm --quiet

In [51]:
import os
import re
import glob
import json
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import librosa
import joblib
import pyloudnorm as pyln

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Wav2Vec2FeatureExtractor,
)
from huggingface_hub import hf_hub_download
from jiwer import wer
from hazm import Normalizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", TORCH_DEVICE)

# Model locations 
STT_MODEL_NAME = "openai/whisper-large-v3"

# Text emotion model, published on the Hugging Face Hub
SENTIMENT_MODEL_PATH = "ftmjfr/VEMO-final-text-emotion-recognition"

# Voice emotion: Phase 1 checkpoint (encoder only) + final deployable classifier
AUDIO_EMOTION_MODEL_PATH = "/kaggle/working/shemo_phase1_checkpoint"
FINAL_CLASSIFIER_PATH = "/kaggle/working/final_classifier_v2"  # v2 classifier

# Shared test dataset: audio + matching transcripts, emotion parsed from filename suffix
DATASET_BASE_PATH = "/kaggle/input/datasets/ftmjfr/audio-transcript/test/test"
AUDIO_DIR = os.path.join(DATASET_BASE_PATH, "audio")
TRANSCRIPT_DIR = os.path.join(DATASET_BASE_PATH, "transcripts")

STT_TARGET_SR = 16000
normalizer = Normalizer()

# Loudness-normalize to -23 LUFS, clip-guard, trim silence -- no denoising.
STT_TARGET_LUFS = -23.0
STT_TRIM_TOP_DB = 30


def _preprocess_for_stt(speech, sr):
    """Loudness-normalize and trim silence."""
    try:
        meter = pyln.Meter(sr)
        loudness = meter.integrated_loudness(speech)
        if np.isfinite(loudness):
            speech = pyln.normalize.loudness(speech, loudness, STT_TARGET_LUFS)
    except Exception as e:
        print(f"Loudness normalization failed, continuing with raw audio: {e}")

    max_val = np.max(np.abs(speech)) if speech.size else 0.0
    if max_val > 1.0:
        speech = speech / max_val * 0.98

    try:
        speech, _ = librosa.effects.trim(speech, top_db=STT_TRIM_TOP_DB)
    except Exception as e:
        print(f"Silence trimming failed, continuing untrimmed: {e}")

    return speech

Using device: cuda


## 3. Build the test manifest (audio + reference transcript + true emotion from filename)

In [52]:
FILENAME_LABEL_MAP = {
    "HAP": "HAPPY",
    "SAD": "SAD",
    "FEA": "FEAR",
    "ANG": "ANGRY",
    "NEU": "NEUTRAL",
    "SUR": "SURPRISE",
}

def parse_emotion_from_filename(filename: str) -> str:
    base = os.path.splitext(os.path.basename(filename))[0]
    suffix = base.split("_")[-1].upper()
    return FILENAME_LABEL_MAP.get(suffix, "UNKNOWN")


# Recursively find audio files inside each speaker subfolder
audio_files = sorted(
    p for p in Path(AUDIO_DIR).glob("speaker_*/*")
    if p.suffix.lower() in (".m4a", ".aac", ".mp3")
)
print(f"Total audio files found: {len(audio_files)}")

# Check whether reference transcripts exist for this dataset
HAS_TRANSCRIPTS = os.path.isdir(TRANSCRIPT_DIR)
if not HAS_TRANSCRIPTS:
    print("No transcripts folder found -- WER will be skipped, "
          "STT output will still be used as input to the text emotion model.")

manifest_rows = []
for audio_path in audio_files:
    speaker_id = os.path.basename(os.path.dirname(audio_path))
    base_name = os.path.splitext(os.path.basename(audio_path))[0]

    true_emotion = parse_emotion_from_filename(audio_path)
    if true_emotion == "UNKNOWN":
        print(f"Warning: could not parse emotion label from {audio_path}, skipping.")
        continue

    reference_text = None
    if HAS_TRANSCRIPTS:
        # transcripts/ is flat -- no per-speaker subfolder, just base_name + ".txt"
        transcript_path = os.path.join(TRANSCRIPT_DIR, base_name + ".txt")
        if os.path.exists(transcript_path):
            with open(transcript_path, "r", encoding="utf-8", errors="ignore") as f:
                reference_text = f.read().strip()
        else:
            print(f"Warning: no transcript found for {audio_path} (looked for {transcript_path})")

    manifest_rows.append({
        "audio_path": audio_path,
        "speaker_id": speaker_id,
        "reference_text": reference_text,
        "true_emotion": true_emotion,
    })

manifest_df = pd.DataFrame(manifest_rows)
print(f"Total valid samples in manifest: {len(manifest_df)}")
manifest_df.head()

Total audio files found: 93
Total valid samples in manifest: 93


,audio_path,speaker_id,reference_text,true_emotion
0,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/001_ANG.m4a,speaker_1,این بی عدالتی واقعا غیرقابل تحمله,ANGRY
1,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/002_NEU.m4a,speaker_1,قبض برق امروز اومد باید برم پرداخت کنم,NEUTRAL
2,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/003_SAD.m4a,speaker_1,هرروز خبرهای تلخی رو میشنوم که حالمو خیلی خیلی خراب میکنه,SAD
3,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/004_HAP.m4a,speaker_1,امروز یکی از بهترین روزهای زندگیم بود,HAPPY
4,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_10/044_SAD.m4a,speaker_10,انگار همه دنیا علیه منه مدام شکست میخورم,SAD


## 4. Load the STT pipeline

In [53]:
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=STT_MODEL_NAME,
    device=DEVICE,
    torch_dtype=torch.float16 if DEVICE >= 0 else torch.float32,
    chunk_length_s=30,
    generate_kwargs={"language": "fa", "task": "transcribe"},
)
print(f"STT pipeline loaded: {STT_MODEL_NAME}")


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


STT pipeline loaded: openai/whisper-large-v3


## 5. Load the text emotion classification model

In [54]:
text_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_PATH)
text_emotion_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_PATH)
text_emotion_model.eval()
text_emotion_model.to(TORCH_DEVICE)

try:
    label_map_path = hf_hub_download(repo_id=SENTIMENT_MODEL_PATH, filename="label_map.json")
    with open(label_map_path, "r", encoding="utf-8") as f:
        label_map = json.load(f)
    text_id2label = {int(k): v for k, v in label_map["id2label"].items()}
except Exception:
    text_id2label = {int(k): v for k, v in text_emotion_model.config.id2label.items()}

print("Text emotion model loaded. Raw label set:", list(text_id2label.values()))


# Check whether the checkpoint's label set is usable
_placeholder_labels = [v for v in text_id2label.values() if v.upper().startswith("LABEL_")]
if _placeholder_labels:
    print("=" * 70)
    print("ERROR: The text-emotion checkpoint has PLACEHOLDER label names:", _placeholder_labels)
    print("This means label_map.json was not found on the Hub repo, AND the model's")
    print("own config.id2label was never set to the real class names when it was saved.")
    print("Every downstream prediction will silently collapse to the same fallback class")
    print("until this is fixed -- see the diagnostic probe cell right below.")
    print("=" * 70)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Text emotion model loaded. Raw label set: ['ANGRY', 'FEAR', 'HAPPY', 'HATE', 'OTHER', 'SAD', 'SURPRISE']


## 6. Load the voice emotion classification model (SHEMO Phase 1 checkpoint)

In [55]:
from transformers import Wav2Vec2Model

# The encoder is used purely as an embedding extractor -- the actual
# classification head is the separate scikit-learn model loaded below.

audio_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(AUDIO_EMOTION_MODEL_PATH)
audio_emotion_model = Wav2Vec2Model.from_pretrained(AUDIO_EMOTION_MODEL_PATH)
audio_emotion_model.eval()
audio_emotion_model.to(TORCH_DEVICE)

AUDIO_TARGET_SR = audio_feature_extractor.sampling_rate
print("Wav2Vec2 encoder loaded for embedding extraction. Sampling rate:", AUDIO_TARGET_SR)

final_scaler = joblib.load(os.path.join(FINAL_CLASSIFIER_PATH, "scaler.joblib"))
final_clf = joblib.load(os.path.join(FINAL_CLASSIFIER_PATH, "logistic_classifier.joblib"))
print("Voice emotion classifier loaded. Classes:", list(final_clf.classes_))

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: /kaggle/working/shemo_phase1_checkpoint
Key                        | Status     |  | 
---------------------------+------------+--+-
classifier.out_proj.bias   | UNEXPECTED |  | 
classifier.dense.weight    | UNEXPECTED |  | 
classifier.out_proj.weight | UNEXPECTED |  | 
classifier.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wav2Vec2 encoder loaded for embedding extraction. Sampling rate: 16000
Voice emotion classifier loaded. Classes: [np.str_('anger'), np.str_('happiness'), np.str_('neutral'), np.str_('sadness')]


## 7. Unified emotion taxonomy

Both models' raw labels are mapped onto the same 4-class taxonomy used in
production (`anger`, `happiness`, `sadness`, `neutral`). The ground truth
parsed from filenames goes through the same mapping (`FEAR` -> sadness,
`SURPRISE` -> neutral), so every sample stays comparable across both
modalities and no samples are dropped.

In [56]:

MANUAL_ID2LABEL_OVERRIDE = {}

if MANUAL_ID2LABEL_OVERRIDE:
    text_id2label = {i: MANUAL_ID2LABEL_OVERRIDE.get(name, name) for i, name in text_id2label.items()}
    print("INFO: Manual label override applied. text_id2label is now:", text_id2label)
else:
    print("WARNING: MANUAL_ID2LABEL_OVERRIDE is still empty. If the placeholder-label")
    print("         ERROR above, every text-emotion prediction below will keep silently collapsing")
    print("         to NEUTRAL until this dict is filled in using the diagnostic probe results.")

# 4-class unified taxonomy: anger, happiness, sadness, neutral
UNIFIED_LABELS = ["anger", "happiness", "sadness", "neutral"]

# Maps the model's raw classes onto the 4 unified classes
TEXT_MODEL_LABEL_MAP = {
    "ANGRY": "anger",
    "HAPPY": "happiness",
    "HAPPINESS": "happiness",
    "SAD": "sadness",
    "SADNESS": "sadness",
    "NEUTRAL": "neutral",
    "HATE": "anger",
    "FEAR": "sadness",
    "SURPRISE": "neutral",
    "OTHER": "neutral",
}

# Maps the voice classifier's raw classes onto the 4 unified classes
AUDIO_MODEL_LABEL_MAP = {
    "anger": "anger",
    "happiness": "happiness",
    "neutral": "neutral",
    "sadness": "sadness",
}

# Ground truth (from filename) also goes through the same mapping, so
# "ANGRY" (filename) and "anger" (model output) always mean the same bucket.
GROUND_TRUTH_LABEL_MAP = TEXT_MODEL_LABEL_MAP  # same 4-class scheme


def unify_text_probs(raw_probs: dict) -> dict:
    """raw_probs: {raw_label_name: probability}. Aggregates into the 4 unified classes."""
    unified = {label: 0.0 for label in UNIFIED_LABELS}
    for raw_label, p in raw_probs.items():
        key = str(raw_label).upper()
        unified[TEXT_MODEL_LABEL_MAP.get(key, "neutral")] += float(p)
    return unified


def unify_audio_probs(raw_probs: dict) -> dict:
    unified = {label: 0.0 for label in UNIFIED_LABELS}
    for raw_label, p in raw_probs.items():
        key = str(raw_label).lower()
        unified[AUDIO_MODEL_LABEL_MAP.get(key, "neutral")] += float(p)
    return unified


_unmapped_text = sorted({l for l in text_id2label.values() if l.upper() not in TEXT_MODEL_LABEL_MAP})
if _unmapped_text:
    print("WARNING: unmapped text model labels (will fall back to neutral):", _unmapped_text)
else:
    print("Text model label check OK.")

_unmapped_audio = sorted({l for l in final_clf.classes_ if l.lower() not in AUDIO_MODEL_LABEL_MAP})
if _unmapped_audio:
    print("WARNING: unmapped voice classifier labels (will fall back to neutral):", _unmapped_audio)
else:
    print("Voice classifier label check OK.")

print("Unified taxonomy (shared by both modalities):", UNIFIED_LABELS)


         ERROR above, every text-emotion prediction below will keep silently collapsing
         to NEUTRAL until this dict is filled in using the diagnostic probe results.
Text model label check OK.
Voice classifier label check OK.
Unified taxonomy (shared by both modalities): ['anger', 'happiness', 'sadness', 'neutral']


## 8. Text normalization and WER helper functions

In [57]:
def clean_text(text: str) -> str:
    """Normalize Persian text before WER computation."""
    if not isinstance(text, str):
        return ""
    text = normalizer.normalize(text)
    text = re.sub(r"[\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)  # remove diacritics
    text = re.sub(r"[^\w\s]", " ", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_wer(reference: str, hypothesis: str):
    ref_clean = clean_text(reference)
    hyp_clean = clean_text(hypothesis)
    if len(ref_clean) == 0 or len(hyp_clean) == 0:
        return None
    return wer(ref_clean, hyp_clean)

print("Helper functions ready.")


Helper functions ready.


## 9. Emotion classification helper functions

In [58]:
def classify_text_emotion(text: str):
    """Returns a dict of unified class probabilities, e.g. {'anger':.., 'happiness':.., ...}."""
    if len(text) == 0:
        return {label: (1.0 if label == "neutral" else 0.0) for label in UNIFIED_LABELS}
    inputs = text_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(TORCH_DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = text_emotion_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    raw_probs = {text_id2label.get(i, str(i)): p for i, p in enumerate(probs)}
    return unify_text_probs(raw_probs)


def extract_prosodic_features(speech, sr):
    """8 hand-crafted prosodic features (f0/RMS/ZCR/spectral centroid stats).
    Must be concatenated with the Wav2Vec2 embedding in the exact same order
    used when final_scaler / final_clf were trained (see metadata.json)."""
    f0, _, _ = librosa.pyin(speech, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
    f0_voiced = f0[~np.isnan(f0)]
    if len(f0_voiced) == 0:
        f0_mean, f0_std, f0_range = 0.0, 0.0, 0.0
    else:
        f0_mean, f0_std = float(np.mean(f0_voiced)), float(np.std(f0_voiced))
        f0_range = float(np.max(f0_voiced) - np.min(f0_voiced))
    rms = librosa.feature.rms(y=speech)[0]
    rms_mean, rms_std = float(np.mean(rms)), float(np.std(rms))
    zcr_mean = float(np.mean(librosa.feature.zero_crossing_rate(speech)[0]))
    centroid = librosa.feature.spectral_centroid(y=speech, sr=sr)[0]
    centroid_mean, centroid_std = float(np.mean(centroid)), float(np.std(centroid))
    return np.array([f0_mean, f0_std, f0_range, rms_mean, rms_std, zcr_mean, centroid_mean, centroid_std])


@torch.no_grad()
def classify_audio_emotion(audio_path: str):
    """Extract a Wav2Vec2 embedding + 8 prosodic features, concatenate them
    (embedding first then prosody -- matching training order), then return a
    dict of unified class probabilities via the fitted logistic regression model."""
    speech, _ = librosa.load(audio_path, sr=AUDIO_TARGET_SR, mono=True)
    inputs = audio_feature_extractor(
        speech, sampling_rate=AUDIO_TARGET_SR, return_tensors="pt", padding=True
    )
    input_values = inputs["input_values"].to(TORCH_DEVICE)

    hidden_states = audio_emotion_model(input_values).last_hidden_state
    embedding = hidden_states.mean(dim=1).squeeze(0).cpu().numpy()

    prosody = extract_prosodic_features(speech, AUDIO_TARGET_SR)
    features = np.concatenate([embedding, prosody]).reshape(1, -1)

    features_scaled = final_scaler.transform(features)
    probs = final_clf.predict_proba(features_scaled)[0]
    raw_probs = {cls: p for cls, p in zip(final_clf.classes_, probs)}
    return unify_audio_probs(raw_probs)

print("Emotion classification helpers ready")


Emotion classification helpers ready


## 10. Run the full pipeline on every sample

For each audio file: transcribe (STT) -> classify emotion from the transcript (text
model) -> classify emotion directly from the audio (voice model) -> compare both
predictions against the true emotion parsed from the filename.


In [59]:
results = []

for i, row in manifest_df.iterrows():
    audio_path = row["audio_path"]
    reference_text = row["reference_text"]
    true_emotion_raw = row["true_emotion"]  # e.g. "ANGRY", "HAPPY", ... from filename
    true_emotion = GROUND_TRUTH_LABEL_MAP.get(true_emotion_raw.upper(), "neutral")

    # STT
    try:
        stt_audio, _ = librosa.load(audio_path, sr=STT_TARGET_SR, mono=True)
        stt_audio = _preprocess_for_stt(stt_audio, STT_TARGET_SR)
        transcript = asr_pipe(stt_audio)["text"].strip()
    except Exception as e:
        print(f"STT error on {audio_path}: {e}")
        transcript = ""

    sample_wer = compute_wer(reference_text, transcript)

    # text-based emotion, full probability vector
    try:
        text_probs = classify_text_emotion(transcript)
    except Exception as e:
        print(f"Text emotion error on {audio_path}: {e}")
        text_probs = {label: (1.0 if label == "neutral" else 0.0) for label in UNIFIED_LABELS}

    # voice-based emotion, full probability vector
    try:
        audio_probs = classify_audio_emotion(audio_path)
    except Exception as e:
        print(f"Voice emotion error on {audio_path}: {e}")
        audio_probs = {label: (1.0 if label == "neutral" else 0.0) for label in UNIFIED_LABELS}

    # fusion: (audio + text) / 2
    combined_probs = {label: (text_probs[label] + audio_probs[label]) / 2 for label in UNIFIED_LABELS}

    text_pred = max(text_probs, key=text_probs.get)
    audio_pred = max(audio_probs, key=audio_probs.get)
    combined_pred = max(combined_probs, key=combined_probs.get)

    row_result = {
        "audio_path": str(audio_path),
        "reference_text": reference_text,
        "transcript": transcript,
        "wer": sample_wer,
        "true_emotion": true_emotion,
        "text_emotion_pred": text_pred,
        "audio_emotion_pred": audio_pred,
        "combined_emotion_pred": combined_pred,
        "text_audio_agree": text_pred == audio_pred,
    }
    for label in UNIFIED_LABELS:
        row_result[f"text_prob_{label}"] = text_probs[label]
        row_result[f"audio_prob_{label}"] = audio_probs[label]
        row_result[f"combined_prob_{label}"] = combined_probs[label]

    results.append(row_result)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(manifest_df)} samples...")

results_df = pd.DataFrame(results)
print(f"\nDone. Total samples processed: {len(results_df)}")
results_df.head()


/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_path, sr=STT_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/638788467.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, _ = librosa.load(audio_path, sr=AUDIO_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_pat

Processed 10/93 samples...


/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_path, sr=STT_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/638788467.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, _ = librosa.load(audio_path, sr=AUDIO_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_pat

Processed 20/93 samples...


/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_path, sr=STT_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/638788467.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, _ = librosa.load(audio_path, sr=AUDIO_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_pat

...

/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_path, sr=STT_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/638788467.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, _ = librosa.load(audio_path, sr=AUDIO_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_pat

Processed 90/93 samples...


/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_path, sr=STT_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/638788467.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, _ = librosa.load(audio_path, sr=AUDIO_TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/301263238.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  stt_audio, _ = librosa.load(audio_pat


Done. Total samples processed: 93


,audio_path,reference_text,transcript,wer,true_emotion,text_emotion_pred,audio_emotion_pred,combined_emotion_pred,text_audio_agree,text_prob_anger,...,combined_prob_anger,text_prob_happiness,audio_prob_happiness,combined_prob_happiness,text_prob_sadness,audio_prob_sadness,combined_prob_sadness,text_prob_neutral,audio_prob_neutral,combined_prob_neutral
0,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/001_ANG.m4a,این بی عدالتی واقعا غیرقابل تحمله,این بیعدالتی واقعا غیر قابل تحمله,0.666667,anger,anger,happiness,anger,False,0.999603,...,0.580672,0.000052,0.803435,0.401743,0.000199,0.013203,0.006701,0.000147,0.021621,0.010884
1,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/002_NEU.m4a,قبض برق امروز اومد باید برم پرداخت کنم,قبض برقه مرزو مرد باید به هم پرداخت کنه.,0.750000,neutral,neutral,neutral,neutral,True,0.003057,...,0.004910,0.000067,0.002600,0.001333,0.000317,0.073674,0.036996,0.996558,0.916964,0.956761
2,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/003_SAD.m4a,هرروز خبرهای تلخی رو میشنوم که حالمو خیلی خیلی خراب میکنه,هر روز خبرهای ترخی رو میشنبن که حال ما خیلی خیلی خراب میکنه,0.583333,sadness,sadness,sadness,sadness,True,0.014304,...,0.008268,0.000165,0.392819,0.196492,0.985267,0.467420,0.726344,0.000264,0.137529,0.068897
3,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/004_HAP.m4a,امروز یکی از بهترین روزهای زندگیم بود,امروز یکی از بهترین روزای زندگیم بود,0.142857,happiness,happiness,happiness,happiness,True,0.000318,...,0.000967,0.999230,0.597248,0.798239,0.000247,0.018839,0.009543,0.000205,0.382296,0.191251
4,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_10/044_SAD.m4a,انگار همه دنیا علیه منه مدام شکست میخورم,اینگاه همه دنیا علای همه نه مدام شکست بخورم,0.666667,sadness,anger,anger,anger,True,0.936722,...,0.837041,0.000381,0.000864,0.000622,0.001051,0.032753,0.016902,0.061846,0.229024,0.145435


In [60]:
# Side-by-side comparison: reference vs predicted, with explicit error flags
pd.set_option("display.max_colwidth", 100)

comparison_df = results_df.copy()

# STT-level flags
comparison_df["stt_empty"] = comparison_df["transcript"].str.strip().eq("")
comparison_df["stt_high_wer"] = comparison_df["wer"].apply(lambda w: w is not None and w > 0.5)
comparison_df["stt_no_reference"] = comparison_df["reference_text"].isna() | (
    comparison_df["reference_text"].astype(str).str.strip() == ""
)

# emotion-level flags
comparison_df["text_emotion_correct"] = comparison_df["text_emotion_pred"] == comparison_df["true_emotion"]
comparison_df["audio_emotion_correct"] = comparison_df["audio_emotion_pred"] == comparison_df["true_emotion"]

# one combined status column to filter/sort on
def build_status(row):
    issues = []
    if row["stt_empty"]:
        issues.append("STT_EMPTY")
    if row["stt_high_wer"]:
        issues.append("STT_HIGH_WER")
    if row["stt_no_reference"]:
        issues.append("NO_REFERENCE")
    if not row["text_emotion_correct"]:
        issues.append("TEXT_EMOTION_WRONG")
    if not row["audio_emotion_correct"]:
        issues.append("AUDIO_EMOTION_WRONG")
    return ", ".join(issues) if issues else "OK"

comparison_df["status"] = comparison_df.apply(build_status, axis=1)

    lambda r: r[f"text_prob_{r['text_emotion_pred']}"], axis=1
)
comparison_df["audio_emotion_confidence"] = comparison_df.apply(
    lambda r: r[f"audio_prob_{r['audio_emotion_pred']}"], axis=1
)

# final side-by-side table, worst rows first
display_cols = [
    "audio_path", "reference_text", "transcript", "wer",
    "true_emotion", "text_emotion_pred", "text_emotion_confidence",
    "audio_emotion_pred", "audio_emotion_confidence", "status",
]
comparison_df = comparison_df.sort_values(
    by="status", key=lambda s: s.eq("OK"),  
).reset_index(drop=True)

print("=" * 70)
print("Reference vs. Predicted - Full Comparison")
print("=" * 70)
print(f"Total samples          : {len(comparison_df)}")
print(f"OK (no issues)          : {(comparison_df['status'] == 'OK').sum()}")
print(f"STT empty transcript   : {comparison_df['stt_empty'].sum()}")
print(f"STT high WER (>0.5)    : {comparison_df['stt_high_wer'].sum()}")
print(f"Missing reference text : {comparison_df['stt_no_reference'].sum()}")
print(f"Text emotion wrong     : {(~comparison_df['text_emotion_correct']).sum()}")
print(f"Audio emotion wrong    : {(~comparison_df['audio_emotion_correct']).sum()}")
print("=" * 70)

comparison_df.to_csv("/kaggle/working/reference_vs_predicted.csv", index=False, encoding="utf-8-sig")
print("Saved full comparison table to /kaggle/working/reference_vs_predicted.csv\n")

comparison_df[display_cols]

Reference vs. Predicted - Full Comparison
Total samples          : 93
OK (no issues)          : 51
STT empty transcript   : 0
STT high WER (>0.5)    : 21
Missing reference text : 0
Text emotion wrong     : 17
Audio emotion wrong    : 24
Saved full comparison table to /kaggle/working/reference_vs_predicted.csv



,audio_path,reference_text,transcript,wer,true_emotion,text_emotion_pred,text_emotion_confidence,audio_emotion_pred,audio_emotion_confidence,status
0,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_1/001_ANG.m4a,این بی عدالتی واقعا غیرقابل تحمله,این بیعدالتی واقعا غیر قابل تحمله,0.666667,anger,anger,0.999603,happiness,0.803435,"STT_HIGH_WER, AUDIO_EMOTION_WRONG"
1,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_18/076_ANG.m4a,وخی صدا این آهنگو کم کن اعصابمونو خورد کردی,مخیصه ده یه آن هنگ چمکون، هست آمون خود چردی,1.111111,anger,anger,0.999202,anger,0.978741,STT_HIGH_WER
2,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_18/077_HAP.m4a,امروز رفتم باشگاه مربی گفت شکمت کوچیک شده خوشحال شدم,امرو رفتم باش که اما رو روی گفتیم کمت که چیک شده خوشحال شدم,1.000000,happiness,happiness,0.998685,anger,0.809024,"STT_HIGH_WER, AUDIO_EMOTION_WRONG"
3,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_18/078_SAD.m4a,ای داد امروز دوباره قهوه امون سوخت,ای داد امردار قهوه مو سوخت,0.428571,sadness,sadness,0.692394,happiness,0.490159,AUDIO_EMOTION_WRONG
4,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_19/079_ANG.m4a,خاک تو سرت با این کابینت درست کردنت,خواهد تو سخت با این کابینت درست کردنه,0.375000,anger,anger,0.998720,happiness,0.634004,AUDIO_EMOTION_WRONG
...,...,...,...,...,...,...,...,...,...,...
88,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_3/010_HAP.m4a,حس سبکی داشتم انگار که هیچ غمی امروز وجود نداره وای خیلی خوشحالم خیلی,هست صباکی داشتم انگار که هیچ قمی امروز وجود نداره وای خیلی خوشحالم خیلی,0.214286,happiness,happiness,0.999122,happiness,0.999965,OK
89,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_12/054_NEU.m4a,امروز تصمیم گرفتم یه فیلم جدید ببینم,امروز تصمیم گرفتم یه فیلم جدید ببینم,0.000000,neutral,neutral,0.999391,neutral,0.864878,OK
90,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_12/053_ANG.m4a,بی نظمی این شهر واقعا اعصاب منو خورد کرده,بی نظمی این شهر واقعا عصب منو خورد کرده,0.111111,anger,anger,0.999633,anger,0.984934,OK
91,/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/audio/speaker_12/052_HAP.m4a,امروز خاطره قشنگی با دوستام ساختم,امروز خاطره یک قشنگی با دوستم ساختم,0.333333,happiness,happiness,0.999250,happiness,0.999957,OK


## 11. STT metrics

In [61]:
valid_wer_rows = results_df.dropna(subset=["wer"])
overall_wer = valid_wer_rows["wer"].mean()
overall_stt_accuracy = (1 - overall_wer) * 100

print("=" * 55)
print("STT Evaluation")
print(f"Valid samples: {len(valid_wer_rows)}")
print(f"Average WER  : {overall_wer * 100:.2f}%")
print(f"Accuracy     : {overall_stt_accuracy:.2f}%")
print("=" * 55)


STT Evaluation
Valid samples: 93
Average WER  : 34.19%
Accuracy     : 65.81%


## 12. Text-based emotion metrics

In [62]:
y_true = results_df["true_emotion"].tolist()
y_pred_text = results_df["text_emotion_pred"].tolist()

text_accuracy = accuracy_score(y_true, y_pred_text) * 100
text_precision, text_recall, text_f1, _ = precision_recall_fscore_support(
    y_true, y_pred_text, labels=UNIFIED_LABELS, average="macro", zero_division=0
)

print("=" * 55)
print("Text-based Emotion Evaluation")
print(f"Accuracy       : {text_accuracy:.2f}%")
print(f"Macro Precision: {text_precision * 100:.2f}%")
print(f"Macro Recall   : {text_recall * 100:.2f}%")
print(f"Macro F1       : {text_f1 * 100:.2f}%")
print("=" * 55)
print(classification_report(y_true, y_pred_text, labels=UNIFIED_LABELS, zero_division=0))


Text-based Emotion Evaluation
Accuracy       : 81.72%
Macro Precision: 85.86%
Macro Recall   : 81.21%
Macro F1       : 81.93%
              precision    recall  f1-score   support

       anger       0.65      0.96      0.77        25
   happiness       0.95      0.87      0.91        23
     sadness       0.93      0.64      0.76        22
     neutral       0.90      0.78      0.84        23

    accuracy                           0.82        93
   macro avg       0.86      0.81      0.82        93
weighted avg       0.85      0.82      0.82        93



## 13. Voice-based emotion metrics


In [63]:
y_true_audio = results_df["true_emotion"].tolist()
y_pred_audio = results_df["audio_emotion_pred"].tolist()

audio_accuracy = accuracy_score(y_true_audio, y_pred_audio) * 100
audio_precision, audio_recall, audio_f1, _ = precision_recall_fscore_support(
    y_true_audio, y_pred_audio, labels=UNIFIED_LABELS, average="macro", zero_division=0
)

print("=" * 55)
print("Voice-based Emotion Evaluation")
print(f"Evaluated samples: {len(results_df)}")
print(f"Accuracy         : {audio_accuracy:.2f}%")
print(f"Macro Precision  : {audio_precision * 100:.2f}%")
print(f"Macro Recall     : {audio_recall * 100:.2f}%")
print(f"Macro F1         : {audio_f1 * 100:.2f}%")
print("=" * 55)
print(classification_report(y_true_audio, y_pred_audio, labels=UNIFIED_LABELS, zero_division=0))


Voice-based Emotion Evaluation
Evaluated samples: 93
Accuracy         : 74.19%
Macro Precision  : 74.74%
Macro Recall     : 73.87%
Macro F1         : 74.01%
              precision    recall  f1-score   support

       anger       0.70      0.84      0.76        25
   happiness       0.70      0.70      0.70        23
     sadness       0.74      0.64      0.68        22
     neutral       0.86      0.78      0.82        23

    accuracy                           0.74        93
   macro avg       0.75      0.74      0.74        93
weighted avg       0.75      0.74      0.74        93



## 14. Agreement between text-based and voice-based emotion predictions

In [64]:
agreement_rate = (results_df["text_emotion_pred"] == results_df["audio_emotion_pred"]).mean() * 100

print("=" * 55)
print("Text vs. Voice Emotion Agreement")
print(f"Samples: {len(results_df)}")
print(f"Agreement rate: {agreement_rate:.2f}%")
print("=" * 55)

print("\nCross-tabulation (rows = text prediction, columns = voice prediction):")
print(pd.crosstab(results_df["text_emotion_pred"], results_df["audio_emotion_pred"]))


Text vs. Voice Emotion Agreement
Samples: 93
Agreement rate: 70.97%

Cross-tabulation (rows = text prediction, columns = voice prediction):
audio_emotion_pred  anger  happiness  neutral  sadness
text_emotion_pred                                     
anger                  25          7        2        3
happiness               4         14        2        1
neutral                 1          1       15        3
sadness                 0          1        2       12


### Fusion formula
Both models return a full probability vector over the same 4 unified classes,
and the combined prediction is the argmax of `(audio_probs + text_probs) / 2`.


In [65]:
fusion_summary_rows = []
for mode, col in [("Text only", "text_emotion_pred"),
                   ("Audio only", "audio_emotion_pred"),
                   ("Combined (audio+text)/2", "combined_emotion_pred")]:
    y_pred = results_df[col].tolist()
    acc = accuracy_score(y_true, y_pred)
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=UNIFIED_LABELS,
                                                    average="macro", zero_division=0)
    fusion_summary_rows.append({"Mode": mode, "Accuracy": acc, "Macro-F1": f1})
    print(f"=== {mode} ===")
    print(classification_report(y_true, y_pred, labels=UNIFIED_LABELS, zero_division=0))
    print(confusion_matrix(y_true, y_pred, labels=UNIFIED_LABELS))
    print()

fusion_summary_df = pd.DataFrame(fusion_summary_rows)
print(fusion_summary_df)


=== Text only ===
              precision    recall  f1-score   support

       anger       0.65      0.96      0.77        25
   happiness       0.95      0.87      0.91        23
     sadness       0.93      0.64      0.76        22
     neutral       0.90      0.78      0.84        23

    accuracy                           0.82        93
   macro avg       0.86      0.81      0.82        93
weighted avg       0.85      0.82      0.82        93

[[24  0  1  0]
 [ 2 20  0  1]
 [ 7  0 14  1]
 [ 4  1  0 18]]

=== Audio only ===
              precision    recall  f1-score   support

       anger       0.70      0.84      0.76        25
   happiness       0.70      0.70      0.70        23
     sadness       0.74      0.64      0.68        22
     neutral       0.86      0.78      0.82        23

    accuracy                           0.74        93
   macro avg       0.75      0.74      0.74        93
weighted avg       0.75      0.74      0.74        93

[[21  3  1  0]
 [ 5 16  1  1]
 

## 15. Save all outputs

In [66]:
results_df.to_csv("/kaggle/working/full_pipeline_results.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame([{
    "num_samples": len(results_df),
    "stt_wer_percent": round(overall_wer * 100, 2),
    "stt_accuracy_percent": round(overall_stt_accuracy, 2),
    "text_emotion_accuracy_percent": round(text_accuracy, 2),
    "text_emotion_macro_f1_percent": round(text_f1 * 100, 2),
    "voice_emotion_accuracy_percent": round(audio_accuracy, 2),
    "voice_emotion_macro_f1_percent": round(audio_f1 * 100, 2),
    "text_voice_agreement_percent": round(agreement_rate, 2),
    "combined_accuracy_percent": round(fusion_summary_df.loc[fusion_summary_df["Mode"] == "Combined (audio+text)/2", "Accuracy"].iloc[0] * 100, 2),
    "combined_macro_f1_percent": round(fusion_summary_df.loc[fusion_summary_df["Mode"] == "Combined (audio+text)/2", "Macro-F1"].iloc[0] * 100, 2),
}])
summary.to_csv("/kaggle/working/full_pipeline_summary.csv", index=False)
fusion_summary_df.to_csv("/kaggle/working/fusion_summary.csv", index=False)

print("Saved output files:")
print(" - /kaggle/working/full_pipeline_results.csv")
print(" - /kaggle/working/full_pipeline_summary.csv")
print(" - /kaggle/working/fusion_summary.csv")


Saved output files:
 - /kaggle/working/full_pipeline_results.csv
 - /kaggle/working/full_pipeline_summary.csv
 - /kaggle/working/fusion_summary.csv
